# 🏴 OZZ + RAPTOR — Unified Autonomous Pentesting Pipeline

**DEF CON 34 AI Village HALctf** + **RAPTOR Security Framework**

Pipeline que combina:
- **Ozz (MNHI 3.5)**: Agente autônomo de pentest ativo (nmap, sqlmap, hydra, reverse shells)
- **RAPTOR**: Análise estática de código (Semgrep), validação de vulnerabilidades, geração de PoC
- **Qwen 2.5 Coder 3B**: Modelo local via FastAPI na porta 8000

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│   OZZ AGENT  │────▶│   BRIDGE     │────▶│   RAPTOR     │
│  (Active)    │     │  (Shared DB) │     │  (Static)    │
│              │     │              │     │              │
│  Recon       │     │  Findings    │     │  Semgrep     │
│  Exploit     │     │  Creds       │     │  Validate    │
│  Flags       │     │  Flags       │     │  PoC Gen     │
└──────────────┘     └──────────────┘     └──────────────┘
       │                    │                    │
       └────────────────────┼────────────────────┘
                            ▼
                    ┌──────────────┐
                    │   RELATÓRIO  │
                    │   UNIFICADO  │
                    └──────────────┘
```

In [ ]:
# Cell 1: Setup — Instalar dependências
!pip install -q fastapi uvicorn pydantic requests transformers torch accelerate semgrep

import os, sys, time, requests, subprocess, json, traceback, sqlite3
from pathlib import Path

os.makedirs('/kaggle/working/hf_cache', exist_ok=True)
WORK = '/kaggle/working'
print('✅ Ambiente configurado!')

In [ ]:
# Cell 2: Clonar Ozz + RAPTOR
!if [ -d {WORK}/ozz-halctf ]; then rm -rf {WORK}/ozz-halctf; fi
!git clone https://github.com/Tretabolt/ozz-halctf.git {WORK}/ozz-halctf 2>&1 | tail -3

!if [ -d {WORK}/raptor ]; then rm -rf {WORK}/raptor; fi
!git clone https://github.com/gadievron/raptor.git {WORK}/raptor 2>&1 | tail -3

# Instalar dependências do Ozz
os.chdir(f'{WORK}/ozz-halctf')
!pip install -q -r requirements.txt 2>&1 | tail -3

print('✅ Repositórios clonados!')

In [ ]:
# Cell 3: Subir Sandbox CTF Node.js na porta 3000
!if [ ! -d {WORK}/ctf-sandbox ]; then git clone https://github.com/kimdane/ctf.git {WORK}/ctf-sandbox || true; fi
!cd {WORK}/ctf-sandbox && (npm install --production --silent || true)

ctf_log = open(f'{WORK}/ctf.log', 'w', encoding='utf-8')
ctf_proc = subprocess.Popen(
    ['npm', 'start'],
    cwd=f'{WORK}/ctf-sandbox',
    env={**os.environ, 'PORT': '3000'},
    stdout=ctf_log,
    stderr=ctf_log
)
print('🚀 Sandbox CTF rodando na porta 3000!')

In [ ]:
# Cell 4: Servidor LLM (Qwen 2.5 Coder 3B) via FastAPI na porta 8000
server_script = '''
import os, sys, torch, traceback
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Dict, Any, Optional, Union
from transformers import AutoModelForCausalLM, AutoTokenizer
import uvicorn

app = FastAPI()
model_id = "Qwen/Qwen2.5-Coder-3B-Instruct"
cache_dir = "/kaggle/working/hf_cache"
print("📥 Carregando modelo Qwen 2.5 3B...", flush=True)

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    current_device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if current_device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(model_id, cache_dir=cache_dir, torch_dtype=dtype, device_map=current_device, trust_remote_code=True)
    print(f"✅ Modelo carregado em {current_device}!", flush=True)
except Exception as e:
    print(f"❌ FALHA: {e}", flush=True)
    sys.exit(1)

class ChatRequest(BaseModel):
    model: str
    messages: List[Dict[str, str]]
    max_tokens: Optional[int] = 1024
    temperature: Optional[float] = 0.3
    stop: Optional[Union[str, List[str]]] = None

@app.get("/v1/models")
def get_models():
    return {"data": [{"id": model_id}, {"id": "qwen2.5-coder-3b"}]}

@app.post("/v1/chat/completions")
def chat_completion(req: ChatRequest):
    try:
        prompt = tokenizer.apply_chat_template(req.messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(current_device)
        max_tok = min(req.max_tokens or 1024, 1024)
        temp = req.temperature if req.temperature is not None else 0.3
        gen_kwargs = {"max_new_tokens": max_tok, "do_sample": temp > 0, "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id}
        if temp > 0: gen_kwargs["temperature"] = temp
        with torch.no_grad():
            outputs = model.generate(**inputs, **gen_kwargs)
        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        return {"choices": [{"message": {"role": "assistant", "content": response_text}}]}
    except Exception as err:
        raise HTTPException(status_code=500, detail=str(err))

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open(f"{WORK}/hf_server.py", "w", encoding="utf-8") as f:
    f.write(server_script)

log_file = open(f"{WORK}/hf_server.log", "w", encoding="utf-8")
server_proc = subprocess.Popen(["python3", "-u", f"{WORK}/hf_server.py"],
    stdout=log_file, stderr=log_file, env={**os.environ, "PYTHONUNBUFFERED": "1"})
print('⚡ Servidor LLM disparado na porta 8000!')

In [ ]:
# Cell 5: Health Check — Aguardar servidores
def wait_for_service(name, url, max_wait=120):
    print(f'⏳ Aguardando {name} ({url})...')
    for i in range(max_wait // 5):
        try:
            r = requests.get(url, timeout=2)
            if r.status_code in [200, 301, 302, 403, 404]:
                print(f'✅ {name} pronto!')
                return True
        except: pass
        time.sleep(5)
    print(f'❌ {name} não respondeu!')
    return False

llm_ok = wait_for_service('Qwen LLM', 'http://localhost:8000/v1/models')
ctf_ok = wait_for_service('CTF Sandbox', 'http://localhost:3000')

if llm_ok:
    # Smoke test
    try:
        r = requests.post('http://localhost:8000/v1/chat/completions',
            json={'model': 'Qwen/Qwen2.5-Coder-3B-Instruct',
                  'messages': [{'role': 'user', 'content': 'Say hello in 3 words'}],
                  'max_tokens': 50}, timeout=60)
        print(f'🧪 LLM Test: {r.json()["choices"][0]["message"]["content"][:100]}')
    except Exception as e:
        print(f'⚠️ LLM smoke test falhou: {e}')

In [ ]:
# Cell 6: 🔬 RAPTOR — Análise estática com Semgrep
print('='*60)
print('🔬 RAPTOR — Static Analysis Phase')
print('='*60)

import subprocess, json

RAPTOR_DIR = f'{WORK}/raptor'
OZZ_DIR = f'{WORK}/ozz-halctf'
REPORTS_DIR = f'{WORK}/reports'
os.makedirs(REPORTS_DIR, exist_ok=True)

# Targets para análise estática (código-fonte dos containers CTF)
static_targets = [
    f'{OZZ_DIR}/universe/target-01',  # Web (LFI + SQLi)
    f'{OZZ_DIR}/universe/target-03',  # Flask API (SSTI + JWT)
]

semgrep_findings = []

for target in static_targets:
    if not os.path.isdir(target):
        print(f'⚠️ Target não encontrado: {target}')
        continue
    
    target_name = os.path.basename(target)
    print(f'\n🔍 Scanning {target_name}...')
    
    # Rodar Semgrep com rules de segurança
    sarif_file = f'{REPORTS_DIR}/{target_name}_semgrep.sarif'
    try:
        result = subprocess.run(
            ['semgrep',
             '--config', 'auto',          # Rules automáticas
             '--config', 'p/owasp-top-ten',  # OWASP Top 10
             '--config', 'p/security-audit',  # Security audit
             '--sarif',
             '--output', sarif_file,
             '--quiet',
             target],
            capture_output=True, text=True, timeout=120
        )
        
        # Parse SARIF output
        if os.path.exists(sarif_file):
            with open(sarif_file) as f:
                sarif = json.load(f)
            
            runs = sarif.get('runs', [])
            for run in runs:
                for result_item in run.get('results', []):
                    rule_id = result_item.get('ruleId', 'unknown')
                    message = result_item.get('message', {}).get('text', '')
                    locations = result_item.get('locations', [])
                    file_path = locations[0].get('physicalLocation', {}).get('artifactLocation', {}).get('uri', '') if locations else ''
                    line = locations[0].get('physicalLocation', {}).get('region', {}).get('startLine', 0) if locations else 0
                    
                    severity = 'medium'
                    for loc in result_item.get('locations', []):
                        level = loc.get('properties', {}).get('severity', '')
                        if 'error' in level or 'critical' in level: severity = 'critical'
                        elif 'warning' in level: severity = 'high'
                    
                    finding = {
                        'source': 'raptor_semgrep',
                        'target': target_name,
                        'rule_id': rule_id,
                        'severity': severity,
                        'message': message,
                        'file': file_path,
                        'line': line
                    }
                    semgrep_findings.append(finding)
                    print(f'  🚨 [{severity.upper()}] {rule_id}: {message[:80]}')
            
            print(f'  📊 {len(run.get("results", []))} findings em {target_name}')
        else:
            print(f'  ⚠️ Sem SARIF output para {target_name}')
            if result.stderr:
                print(f'  Stderr: {result.stderr[:200]}')
    except subprocess.TimeoutExpired:
        print(f'  ⏰ Timeout scanning {target_name}')
    except Exception as e:
        print(f'  ❌ Erro: {e}')

# Salvar findings consolidados
with open(f'{REPORTS_DIR}/semgrep_findings.json', 'w') as f:
    json.dump(semgrep_findings, f, indent=2)

print(f'\n✅ RAPTOR: {len(semgrep_findings)} findings totais')
print(f'📄 Relatório salvo em {REPORTS_DIR}/semgrep_findings.json')

In [ ]:
# Cell 7: 🧠 Qwen analisa os findings do RAPTOR e gera PoCs
print('='*60)
print('🧠 LLM Analysis — Qwen analisa findings do RAPTOR')
print('='*60)

LLM_URL = 'http://localhost:8000/v1/chat/completions'
LLM_MODEL = 'Qwen/Qwen2.5-Coder-3B-Instruct'

def ask_qwen(prompt, max_tokens=1024, temperature=0.2):
    """Chama o Qwen para análise."""
    try:
        r = requests.post(LLM_URL, json={
            'model': LLM_MODEL,
            'messages': [{'role': 'user', 'content': prompt}],
            'max_tokens': max_tokens,
            'temperature': temperature
        }, timeout=120)
        return r.json()['choices'][0]['message']['content']
    except Exception as e:
        return f'ERROR: {e}'

# Carregar findings
with open(f'{REPORTS_DIR}/semgrep_findings.json') as f:
    all_findings = json.load(f)

# Analisar cada finding com o Qwen
analyzed_findings = []

for finding in all_findings[:10]:  # Limitar a 10 para não estourar tokens
    prompt = f"""You are a security expert. Analyze this vulnerability finding and:
1. Assess if it's a TRUE POSITIVE or FALSE POSITIVE
2. Rate severity (critical/high/medium/low)
3. Explain the exploitability
4. Generate a Proof of Concept (PoC) if exploitable

Finding:
- Rule: {finding['rule_id']}
- File: {finding['file']}:{finding['line']}
- Description: {finding['message']}
- Target: {finding['target']}

Respond in JSON format:
{{"verdict": "true_positive|false_positive", "severity": "critical|high|medium|low", "exploitability": "...", "poc": "...", "recommendation": "..."}}
"""
    
    analysis = ask_qwen(prompt)
    finding['llm_analysis'] = analysis
    analyzed_findings.append(finding)
    print(f'  ✅ Analisado: {finding["rule_id"]}')

# Salvar análise
with open(f'{REPORTS_DIR}/analyzed_findings.json', 'w') as f:
    json.dump(analyzed_findings, f, indent=2)

print(f'\n📊 {len(analyzed_findings)} findings analisados pelo Qwen')

In [ ]:
# Cell 8: 🏴 OZZ — Agente autônomo de pentest ativo
print('='*60)
print('🏴 OZZ — Active Pentesting Phase')
print('='*60)

os.chdir(f'{WORK}/ozz-halctf')

# Executar o agente Ozz
env_vars = {
    **os.environ,
    'MAX_ITERATIONS': '20',
    'LLM_URL': 'http://localhost:8000/v1',
    'TARGET_URL': 'http://localhost:3000'
}

print('🚀 Disparando agente Ozz MNHI 3.5...')
agent_result = subprocess.run(
    ['python3', '-m', 'agent', 'http://localhost:3000'],
    env=env_vars,
    capture_output=True,
    text=True,
    timeout=1800
)

print(f'Saída do agente:\n{agent_result.stdout[-2000:]}')
if agent_result.stderr:
    print(f'Erros:\n{agent_result.stderr[-1000:]}')

print(f'\n🏁 Ozz finalizou com código: {agent_result.returncode}')

In [ ]:
# Cell 9: 📊 Relatório Unificado — RAPTOR + OZZ
print('='*60)
print('📊 RELATÓRIO UNIFICADO — RAPTOR + OZZ')
print('='*60)

# Carregar todos os resultados
raptor_findings = []
ozz_results = {}

# RAPTOR findings
try:
    with open(f'{REPORTS_DIR}/analyzed_findings.json') as f:
        raptor_findings = json.load(f)
except: pass

# Ozz results (memory/SQLite)
try:
    db_path = f'{WORK}/ozz-halctf/memory/ozz.db'
    if os.path.exists(db_path):
        conn = sqlite3.connect(db_path)
        cursor = conn.execute('SELECT * FROM findings ORDER BY timestamp DESC LIMIT 20')
        columns = [d[0] for d in cursor.description]
        ozz_results['findings'] = [dict(zip(columns, row)) for row in cursor.fetchall()]
        
        cursor = conn.execute('SELECT * FROM flags ORDER BY timestamp DESC')
        columns = [d[0] for d in cursor.description]
        ozz_results['flags'] = [dict(zip(columns, row)) for row in cursor.fetchall()]
        conn.close()
except Exception as e:
    print(f'⚠️ Não foi possível ler memória do Ozz: {e}')

# Gerar relatório consolidado
report = {
    'pipeline': 'OZZ + RAPTOR',
    'model': 'Qwen 2.5 Coder 3B',
    'raptor': {
        'static_findings': len(raptor_findings),
        'critical': len([f for f in raptor_findings if f.get('severity') == 'critical']),
        'high': len([f for f in raptor_findings if f.get('severity') == 'high']),
        'medium': len([f for f in raptor_findings if f.get('severity') == 'medium']),
        'findings': raptor_findings
    },
    'ozz': {
        'flags_found': len(ozz_results.get('flags', [])),
        'flags': ozz_results.get('flags', []),
        'active_findings': len(ozz_results.get('findings', [])),
        'findings': ozz_results.get('findings', [])
    }
}

# Salvar
report_path = f'{REPORTS_DIR}/unified_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)

# Imprimir resumo
print(f"\n{'='*50}")
print(f"  🔬 RAPTOR (Static Analysis)")
print(f"     Total findings: {report['raptor']['static_findings']}")
print(f"     🔴 Critical: {report['raptor']['critical']}")
print(f"     🟠 High:     {report['raptor']['high']}")
print(f"     🟡 Medium:   {report['raptor']['medium']}")
print(f"\n  🏴 OZZ (Active Pentest)")
print(f"     Flags found: {report['ozz']['flags_found']}")
print(f"     Active findings: {report['ozz']['active_findings']}")
for flag in report['ozz']['flags']:
    print(f"     🚩 {flag}")
print(f"{'='*50}")
print(f"\n📄 Relatório completo: {report_path}")

In [ ]:
# Cell 10: Cleanup
print('🧹 Limpando recursos...')

# Parar servidores
if 'server_proc' in globals():
    server_proc.terminate()
    print('  ⏹ Servidor LLM parado')

if 'ctf_proc' in globals() and ctf_proc.poll() is None:
    ctf_proc.terminate()
    print('  ⏹ Sandbox CTF parada')

if 'log_file' in globals() and not log_file.closed:
    log_file.close()
if 'ctf_log' in globals() and not ctf_log.closed:
    ctf_log.close()

print('\n🏴 Pipeline OZZ + RAPTOR finalizado!')